# 🎯 Gün 1 — Görüntü İşleme Temelleri
**Endüstriyel Bilgisayarlı Görü Eğitimi**

---
## 🔧 K-01 | Ortam Kurulumu & Kütüphaneler
📌 *Slayt 13 — Kaggle Notebook Kurulumu*

In [ ]:
# ================================================================
# K-01: Kütüphane Yükleme & Ortam Kontrolü
# ================================================================

import cv2          # OpenCV — görüntü işleme ana kütüphanesi
import numpy as np  # NumPy  — sayısal matris işlemleri
import matplotlib.pyplot as plt  # Matplotlib — notebook görseli
import os           # Dosya/klasör işlemleri

print(f"OpenCV sürümü : {cv2.__version__}")
print(f"NumPy sürümü  : {np.__version__}")

print("\n📂 Kaggle dizin yapısı:")
print("  /kaggle/input/   → veri setleri (sadece okuma)")
print("  /kaggle/working/ → çıktı dosyaları (okuma + yazma)")

# Yüklü dataset'leri listele
input_path = "/kaggle/input"
if os.path.exists(input_path) and os.listdir(input_path):
    print("\n📦 Dataset'ler:")
    for ds in os.listdir(input_path):
        print(f"   - {ds}")
else:
    print("\n  (Dataset yüklenmemiş — sağ panelde 'Add Data' ile ekle)")

# Matplotlib default ayarları
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 11

print("\n✅ Kurulum tamamlandı!")

In [ ]:
# ================================================================
# K-01-B: Test görseli oluştur
# (Gerçek çalışmada bu bloğu sil — imread ile kendi görselin yükle)
# ================================================================

# NumPy ile sentetik test görseli
test_img = np.zeros((300, 400, 3), dtype=np.uint8)  # Siyah BGR görüntü (H=300, W=400)

# Farklı renkli bölgeler (BGR sırası!)
test_img[20:130,  20:190]  = [255,  0,  0]   # Mavi dikdörtgen  (B=255, G=0,   R=0)
test_img[20:130, 200:380]  = [  0,200,  0]   # Yeşil dikdörtgen (B=0,   G=200, R=0)
test_img[150:280, 20:380]  = [  0,  0,220]   # Kırmızı bant    (B=0,   G=0,   R=220)
test_img[90:200,  150:250] = [  0,220,220]   # Sarı alan       (B=0,   G=220, R=220)

# Hafif gürültü (gerçek görüntüye benzetmek için)
noise = np.random.randint(0, 30, test_img.shape, dtype=np.uint8)
test_img = cv2.add(test_img, noise)  # cv2.add: taşmayı önler (clamp 0-255)

print(f"Test görüntüsü → shape: {test_img.shape}, dtype: {test_img.dtype}")
print("  shape = (H=300, W=400, C=3)  |  dtype = uint8 (piksel: 0-255)")

plt.figure(figsize=(6,4))
plt.imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))  # BGR→RGB! Matplotlib için
plt.title('Test Görseli (K-01)'); plt.axis('off'); plt.show()

img = test_img.copy()  # Tüm notebook boyunca 'img' kullanacağız
print("\n'img' hazır — bundan sonra bu değişkeni kullanıyoruz.")

---
## 📷 K-02 | imread / imwrite / Görüntü Temsili
📌 *Slayt 15–16 — cv2.imread(), cv2.imwrite(), NumPy dizisi*

### 🔑 Temel Kavram: Görüntü = NumPy Matrisi
OpenCV bir görüntüyü 3 boyutlu NumPy dizisi olarak saklar: `(Yükseklik, Genişlik, Kanal)`

In [ ]:
# ================================================================
# K-02-A: cv2.imread() — Parametre detayları
# ================================================================
#
# cv2.imread(filename, flags=cv2.IMREAD_COLOR)
#
#  filename → Dosya yolu (string)
#  flags    → Renk modu:
#
#   cv2.IMREAD_COLOR      (= 1, varsayılan)
#     → 3 kanallı BGR görüntü döner
#     → Transparan (alpha) kanalı yok sayar
#
#   cv2.IMREAD_GRAYSCALE  (= 0)
#     → 1 kanallı gri ton görüntü döner
#     → Bellekte 3x daha az yer    |    3x daha hızlı işlem
#     → Edge/contour/threshold için ideal
#
#   cv2.IMREAD_UNCHANGED  (= -1)
#     → 3 veya 4 kanallı döner (PNG'de alpha kanalı dahil)
#     → Şeffaf PNG'lerde gerekli
#
# ⚠️ UYARI: imread başarısız olunca None döner, hata FIRLATMAZ!
#    Her zaman: assert img is not None, "Dosya bulunamadı!"
#

# Önce test görüntüsünü diske kaydet (okumak için)
test_path = '/kaggle/working/test_img.jpg'
cv2.imwrite(test_path, img)

# --- Farklı flags ile okuma ---
img_color = cv2.imread(test_path, cv2.IMREAD_COLOR)    # 3 kanal BGR
img_gray  = cv2.imread(test_path, cv2.IMREAD_GRAYSCALE) # 1 kanal

# ⚠️ None kontrolü — her zaman yap!
assert img_color is not None, f"Hata: Dosya okunamadı → {test_path}"
assert img_gray  is not None

print("imread sonuçları:")
print(f"  IMREAD_COLOR      → shape: {img_color.shape}  dtype: {img_color.dtype}")
print(f"  IMREAD_GRAYSCALE  → shape: {img_gray.shape}   dtype: {img_gray.dtype}")

# --- NumPy dizisi olarak görüntü ---
print("\nNumPy operasyonları:")
print(f"  img.shape  = {img_color.shape}")
print(f"  img.shape[0] = {img_color.shape[0]}  (H = yükseklik = satır sayısı)")
print(f"  img.shape[1] = {img_color.shape[1]}  (W = genişlik  = sütun sayısı)")
print(f"  img.shape[2] = {img_color.shape[2]}  (C = kanal sayısı: BGR=3, Gray=1)")

# --- Piksel erişimi ---
# ⚠️ img[y, x] — önce y (satır/yükseklik), sonra x (sütun/genişlik)!
y, x = 20, 30
pixel_bgr = img_color[y, x]      # Tüm kanallar: [B, G, R]
pixel_b   = img_color[y, x, 0]   # Sadece Mavi  (indeks 0)
pixel_g   = img_color[y, x, 1]   # Sadece Yeşil (indeks 1)
pixel_r   = img_color[y, x, 2]   # Sadece Kırmızı (indeks 2)
print(f"\nPiksel ({y},{x}): BGR={pixel_bgr}  B={pixel_b} G={pixel_g} R={pixel_r}")

# --- ROI (Region of Interest) kırpma ---
# Sözdizimi: img[y1:y2, x1:x2]  (y aralığı : x aralığı)
roi_ref  = img_color[50:150, 50:200]        # REFERANS — değiştirirse img değişir!
roi_copy = img_color[50:150, 50:200].copy() # KOPYA — bağımsız çalışma
print(f"\nROI boyutu: {roi_ref.shape}  (y:50-150, x:50-200 → 100×150 piksel)")

# Parlaklık artırma (numpy ile)
# np.clip: değerleri [0,255] arasına sıkıştırır → taşmayı önler
brighter = np.clip(img_color.astype(int) + 40, 0, 255).astype(np.uint8)
print(f"\nParlaklık +40 → dtype: {brighter.dtype}  max: {brighter.max()}")

In [ ]:
# ================================================================
# K-02-B: BGR vs RGB — En Sık Yapılan Hata
# ================================================================
#
# OpenCV → BGR sırası: indeks 0=Mavi, 1=Yeşil, 2=Kırmızı
# Matplotlib → RGB sırası bekler
#
# Matplotlib'e BGR görüntü verirsen:
#   Mavi ↔ Kırmızı yer değiştirir  → görüntü yanlış renkli görünür!
#

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ❌ Yanlış — BGR doğrudan imshow'a
axes[0].imshow(img_color)
axes[0].set_title('❌ YANLIŞ: BGR → imshow\nMavi↔Kırmızı yer değiştirdi!', color='red')
axes[0].axis('off')

# ✅ Doğru — BGR→RGB çevirip imshow'a
img_rgb = cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB)
axes[1].imshow(img_rgb)
axes[1].set_title('✅ DOĞRU: BGR→RGB → imshow\ncvtColor ile düzeltildi', color='green')
axes[1].axis('off')

# Gri ton
axes[2].imshow(img_gray, cmap='gray')  # cmap='gray' tek kanallı için şart!
axes[2].set_title('Gri Ton (cmap="gray" ile)')
axes[2].axis('off')

plt.suptitle('K-02: BGR vs RGB — Matplotlib Gösterim Farkı', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("🔑 Kural: matplotlib.plt.imshow() için HER ZAMAN BGR→RGB dönüşümü yap!")

In [ ]:
# ================================================================
# K-02-C: cv2.imwrite() — Parametre detayları
# ================================================================
#
# cv2.imwrite(filename, img, params=[])
#
#  filename → Kaydedilecek dosya yolu (.jpg/.png/.bmp/.tiff)
#             Uzantı, hangi formatın kullanılacağını belirler!
#
#  img      → Kaydedilecek NumPy array
#             OpenCV BGR sırasıyla saklar (çevirme gerekmez!)
#
#  params   → Format-özel kalite/sıkıştırma ayarları
#
#  JPEG: [cv2.IMWRITE_JPEG_QUALITY, kalite]  → kalite: 0-100
#        0   = en düşük kalite, en küçük dosya
#        95  = yüksek kalite (öneri)
#        100 = maksimum kalite (büyük dosya)
#
#  PNG:  [cv2.IMWRITE_PNG_COMPRESSION, seviye]  → 0-9
#        0 = sıkıştırma yok (büyük dosya, hızlı)
#        3 = dengeli (öneri)
#        9 = maksimum sıkıştırma (küçük dosya, yavaş)
#

files = [
    ('/kaggle/working/kalite_95.jpg', [cv2.IMWRITE_JPEG_QUALITY, 95]),   # yüksek kalite
    ('/kaggle/working/kalite_50.jpg', [cv2.IMWRITE_JPEG_QUALITY, 50]),   # orta kalite
    ('/kaggle/working/kalite_10.jpg', [cv2.IMWRITE_JPEG_QUALITY, 10]),   # düşük kalite
    ('/kaggle/working/img.png',       [cv2.IMWRITE_PNG_COMPRESSION, 3]), # PNG kayıpsız
]

for path, params in files:
    cv2.imwrite(path, img_color, params)

print("Dosya boyutu karşılaştırması:")
for path, _ in files:
    size_kb = os.path.getsize(path) / 1024
    print(f"  {os.path.basename(path):<22}: {size_kb:6.1f} KB")

print("\n💡 Genel öneri: Görselleştirme için JPEG 90-95 | Tekrar işlenecekse PNG")

---
## 🎨 K-03 | Renk Uzayları & cv2.cvtColor()
📌 *Slayt 17–18 — BGR • Gray • HSV • LAB*

In [ ]:
# ================================================================
# K-03-A: cv2.cvtColor() — Renk Uzayı Dönüşümü
# ================================================================
#
# cv2.cvtColor(src, code)
#
#  src  → Giriş görüntüsü (NumPy array)
#  code → Dönüşüm kodu (cv2.COLOR_*)
#
# EN ÇOK KULLANILAN KODLAR:
#
#  cv2.COLOR_BGR2RGB   → Matplotlib için (renkler doğru gösterilir)
#  cv2.COLOR_BGR2GRAY  → Edge / blur / threshold öncesi (1 kanal)
#  cv2.COLOR_BGR2HSV   → Renk filtreleme (inRange) için
#  cv2.COLOR_BGR2LAB   → Zor ışık koşulları / renk analizi
#  cv2.COLOR_GRAY2BGR  → Gri görüntüye renkli çizim yapmak için
#  cv2.COLOR_HSV2BGR   → HSV'de düzenle, BGR'a geri dön
#
# RENK UZAYLARI:
#
#  BGR  → B=Mavi[0-255]  G=Yeşil[0-255]  R=Kırmızı[0-255]
#         OpenCV varsayılanı! imread her zaman BGR döner
#
#  GRAY → Tek kanal [0-255]. 0=siyah, 255=beyaz
#         Formül: Gray = 0.114×B + 0.587×G + 0.299×R
#
#  HSV  → H=Ton[0-179]  S=Doygunluk[0-255]  V=Parlaklık[0-255]
#         ⚠️ OpenCV H aralığı 0-179 (Derece÷2)! Normal 0-360 değil!
#         Renk filtreleme için ideal: aydınlatma değişse H stabil
#
#  LAB  → L=Parlaklık[0-255]  A=Yeşil↔Kırmızı  B=Mavi↔Sarı
#         İnsan gözüne yakın algı modeli
#

img_rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)   # Görüntülemek için
img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)  # 3 kanal → 1 kanal
img_hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)   # Renk filtreleme
img_lab  = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)   # Zor ışık koşulları

print("Dönüşüm sonrası shape ve dtype:")
print(f"  BGR  : {img.shape}       {img.dtype}")
print(f"  GRAY : {img_gray.shape}    {img_gray.dtype}  ← 1 kanal, (H,W) yeterli")
print(f"  HSV  : {img_hsv.shape}      {img_hsv.dtype}")
print(f"  LAB  : {img_lab.shape}      {img_lab.dtype}")

# Karşılaştırma
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

panels_row1 = [
    (img_rgb,              'BGR→RGB (Doğru renk)',   None),
    (img_gray,             'GRAY',                   'gray'),
    (img_hsv[:,:,0],       'HSV → H Kanalı\nTon (0–179)', 'hsv'),
    (img_hsv[:,:,1],       'HSV → S Kanalı\nDoygunluk', 'gray'),
]
panels_row2 = [
    (img_hsv[:,:,2],       'HSV → V Kanalı\nParlaklık', 'gray'),
    (img_lab[:,:,0],       'LAB → L Kanalı\nLightness', 'gray'),
    (img_lab[:,:,1],       'LAB → A Kanalı\nYeşil↔Kırmızı', None),
    (img_lab[:,:,2],       'LAB → B Kanalı\nMavi↔Sarı', None),
]

for ax, (im, title, cmap) in zip(axes[0], panels_row1):
    ax.imshow(im, cmap=cmap); ax.set_title(title, fontsize=9); ax.axis('off')
for ax, (im, title, cmap) in zip(axes[1], panels_row2):
    ax.imshow(im, cmap=cmap); ax.set_title(title, fontsize=9); ax.axis('off')

plt.suptitle('K-03: Renk Uzayı Kanalları Karşılaştırması', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# K-03-B: HSV Aralıkları — Hangi renk, hangi H değeri?
# ================================================================
#
# OpenCV HSV'de H aralığı 0–179 (normal 0°–360°'nin YARISI)
# Renk filtresi için lower/upper belirlerken bunu unutma!
#
#  Kırmızı: H=0–10  VE  H=160–179  (renk dairesinin iki ucunda)
#  Turuncu: H=11–25
#  Sarı:    H=26–34
#  Yeşil:   H=35–77
#  Cyan:    H=78–99
#  Mavi:    H=100–130
#  Mor:     H=131–159
#

renkler = {
    'Kırmızı' : (0,   0,   255),  # BGR
    'Turuncu' : (0,   127, 255),
    'Sarı'    : (0,   255, 255),
    'Yeşil'   : (0,   200,  50),
    'Cyan'    : (200, 200,   0),
    'Mavi'    : (200,  0,    0),
    'Mor'     : (180,  0,  180),
}

print(f"{'Renk':<10} {'BGR':<22} {'H':>5}  {'S':>5}  {'V':>5}")
print("-" * 50)

for name, bgr in renkler.items():
    pixel = np.uint8([[list(bgr)]])               # 1×1 piksel array
    hsv   = cv2.cvtColor(pixel, cv2.COLOR_BGR2HSV)
    h, s, v = hsv[0, 0]
    print(f"{name:<10} {str(bgr):<22} {h:>5}  {s:>5}  {v:>5}")

print("\n💡 Kural: Filtre yazarken ±10 tolerans bırak")
print("   lower = [H-10, 50, 50]   upper = [H+10, 255, 255]")
print("   S_min=50 → çok soluk renkleri dışla")
print("   V_min=50 → çok karanlık bölgeleri dışla")

---
## 🌫️ K-04 | Blur & Gürültü Azaltma
📌 *Slayt 20–22 — GaussianBlur • medianBlur • bilateralFilter*

**Neden blur?** → Edge detection öncesinde gürültüyü azaltmak için zorunlu.

In [ ]:
# ================================================================
# K-04-A: cv2.GaussianBlur() — Parametre Detayları
# ================================================================
#
# cv2.GaussianBlur(src, ksize, sigmaX, sigmaY=0, borderType=...)
#
#  src    → Giriş görüntüsü (renkli veya gri)
#
#  ksize  → Kernel boyutu: (genişlik, yükseklik) tuple
#           ⚠️ HER İKİ DEĞER DE TEK SAYI OLMALI!
#           (3,3)  = hafif blur — gürültüyü az azaltır
#           (7,7)  = orta blur  — genel kullanım
#           (15,15)= güçlü blur — gürültü çok yoğunsa
#           (4,4)  = HATA! Çift sayı kabul edilmez
#
#  sigmaX → X yönü Gaussian standart sapması
#           0 → ksize'dan otomatik hesapla (önerilen!)
#           Büyük sigmaX = daha yumuşak geçiş = daha bulanık
#
#  sigmaY → 0 ise sigmaX aynısı kullanılır
#
# ÇALIŞMA PRENSİBİ:
#   Her piksel için çevresini kapsayan kernel boyutunda
#   Gaussian ağırlıklı ortalama alınır.
#   Merkez piksele yüksek, uzak piksellere düşük ağırlık.
#

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Farklı kernel boyutları
g3  = cv2.GaussianBlur(gray, (3, 3),  0)   # hafif
g7  = cv2.GaussianBlur(gray, (7, 7),  0)   # orta        # <-- DEĞİŞTİR
g15 = cv2.GaussianBlur(gray, (15, 15), 0)  # güçlü

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (im, title) in zip(axes, [
    (gray, 'Orijinal'),
    (g3,  'GaussianBlur (3,3)\nHafif'),
    (g7,  'GaussianBlur (7,7)\nOrta'),
    (g15, 'GaussianBlur (15,15)\nGüçlü'),
]):
    ax.imshow(im, cmap='gray'); ax.set_title(title); ax.axis('off')

plt.suptitle('K-04-A: GaussianBlur Kernel Boyutu Etkisi', fontweight='bold')
plt.tight_layout()
plt.show()

print("🔑 Kural: Edge detection'dan ÖNCE mutlaka blur uygula!")
print("   Gürültü → yanlış edge → bozuk contour")

In [ ]:
# ================================================================
# K-04-B: medianBlur & bilateralFilter — Parametre Detayları
# ================================================================
#
# ── cv2.medianBlur(src, ksize) ──────────────────────────────────
#
#  src   → Giriş görüntüsü (uint8)
#  ksize → TEK SAYI (3, 5, 7, 9, 11 ...)
#          ⚠️ ÇİFT SAYI (4,6,8...) HATA VERİR!
#
#  PRENSİP: Penceredeki pikselleri sırala → ortancayı (median) al
#  SONUÇ: Tek noktacıklı gürültü (%100) yok edilir
#  İDEAL: "Tuz-biber" (salt & pepper) gürültüsü
#
# ── cv2.bilateralFilter(src, d, sigmaColor, sigmaSpace) ─────────
#
#  src        → Giriş görüntüsü (uint8)
#
#  d          → Komşu piksel çapı
#               -1  → sigmaSpace'ten otomatik hesapla
#                5  → hızlı (gerçek zamanlı için)
#                9  → dengeli (öneri)
#               15  → yavaş ama güçlü etki
#               ⚠️ d>15 gerçek zamanlı işleme için çok yavaş!
#
#  sigmaColor → Renk benzerlik toleransı (10–150)
#               10  = seçici (sadece çok benzer renkleri karıştır)
#               75  = dengeli (öneri)
#              150  = agresif (farklı renkleri de karıştırır)
#
#  sigmaSpace → Mesafe etkisi — uzaktaki piksellerin katılımı
#               10  = küçük alan etkisi
#               75  = geniş alan (öneri)
#
#  ÖZEL: Kenarları KORUYARAK düz alanları yumuşatır
#

# Tuz-biber gürültüsü oluştur
gray_sp = gray.copy()
mask = np.random.rand(*gray.shape)
gray_sp[mask < 0.05] = 0    # %5 siyah nokta
gray_sp[mask > 0.95] = 255  # %5 beyaz nokta

median5 = cv2.medianBlur(gray_sp, 5)                          # <-- DEĞİŞTİR: 3, 7, 9
bilat   = cv2.bilateralFilter(gray_sp, d=9,
                               sigmaColor=75, sigmaSpace=75)  # <-- DEĞİŞTİR
gaussian= cv2.GaussianBlur(gray_sp, (5,5), 0)                 # Karşılaştırma için

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (im, title) in zip(axes, [
    (gray_sp,  'Tuz-biber gürültüsü'),
    (gaussian, 'GaussianBlur (5,5)\nkenarlar biraz bozulur'),
    (median5,  'medianBlur (k=5)\nnokta gürültüsü temizlendi'),
    (bilat,    'bilateralFilter (d=9)\nkenarlar korundu'),
]):
    ax.imshow(im, cmap='gray'); ax.set_title(title, fontsize=9); ax.axis('off')

plt.suptitle('K-04-B: Blur Türleri Karşılaştırması', fontweight='bold')
plt.tight_layout()
plt.show()

print("📊 Seçim Rehberi:")
print("  Genel gürültü → GaussianBlur")
print("  Tuz-biber     → medianBlur")
print("  Kenar koruyucu→ bilateralFilter (yavaş, dikkatli kullan)")

---
## 📊 K-05 | Kontrast — cv2.equalizeHist()
📌 *Slayt 23 — Histogram Eşitleme*

In [ ]:
# ================================================================
# K-05: cv2.equalizeHist() — Histogram Eşitleme
# ================================================================
#
# cv2.equalizeHist(src)
#
#  src → ⚠️ SADECE 1 kanallı (gri ton) görüntü alır!
#          BGR verirsen hata verir veya yanlış sonuç!
#
#  PRENSİP:
#    Piksel yoğunluk histogramını alır
#    CDF (Cumulative Distribution Function) hesaplar
#    Yoğunlukları 0–255 aralığına dengeli YAYAR
#    → Düşük kontrastlı görüntüler daha belirgin hale gelir
#
#  ⚠️ Global eşitleme → tüm görüntüye tek formül
#     Gün 2'de: CLAHE = Bölgesel adaptif eşitleme → daha iyi!
#

# Düşük kontrastlı gri görüntü simülasyonu
# (Sanki loş ışıkta çekilmiş çelik yüzeyi)
low_c = np.random.randint(60, 160, (300, 400), dtype=np.uint8)
low_c[120:130, :]  = 40   # Koyu çizgi (defect benzeri)
low_c[180:183, 50:350] = 200  # Parlak çizgi

equalized = cv2.equalizeHist(low_c)  # ← sadece gri!

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0,0].imshow(low_c, cmap='gray', vmin=0, vmax=255)
axes[0,0].set_title('Öncesi: Düşük Kontrast\n(piksel 60–160 arası)')
axes[0,0].axis('off')

axes[0,1].imshow(equalized, cmap='gray', vmin=0, vmax=255)
axes[0,1].set_title('Sonrası: equalizeHist\n(piksel 0–255 yayıldı)')
axes[0,1].axis('off')

axes[1,0].hist(low_c.flatten(),  bins=256, range=(0,255), color='steelblue',  edgecolor='none')
axes[1,0].set_title('Histogram — Öncesi'); axes[1,0].set_xlabel('Piksel Değeri')

axes[1,1].hist(equalized.flatten(), bins=256, range=(0,255), color='forestgreen', edgecolor='none')
axes[1,1].set_title('Histogram — Sonrası'); axes[1,1].set_xlabel('Piksel Değeri')

plt.suptitle('K-05: cv2.equalizeHist() — Histogram Eşitleme', fontweight='bold')
plt.tight_layout()
plt.show()

print("Öncesi piksel min/max:", low_c.min(), "/", low_c.max())
print("Sonrası piksel min/max:", equalized.min(), "/", equalized.max())
print("\n📌 Gün 2'de CLAHE öğreneceğiz → adaptif bölgesel eşitleme → daha iyi sonuç!")

---
## 🔨 Mini Proje 1 — Çelik Yüzeyinde Gürültü Azaltma
📌 *Slayt 24 — K-05 | Session 1 Uygulaması (30–40 dk)*

In [ ]:
# ================================================================
# MİNİ PROJE 1: Çelik Yüzey Görüntüsü Ön İşleme
# ================================================================
#
# Önerilen Dataset: NEU Surface Defect Database
# Kaggle: https://www.kaggle.com/datasets/kaustubhdikshit/
#          neu-surface-defect-database
#
# Gerçek kullanım (dataset yüklendikten sonra):
# steel = cv2.imread('/kaggle/input/.../crazing_1.bmp')
# assert steel is not None
#

# Simülasyon: Çelik doku benzeri gürültülü görüntü
np.random.seed(42)
steel_base = np.random.normal(128, 35, (256, 256)).clip(0, 255).astype(np.uint8)
# Çizgisel defect ekle
steel_base[100:104, :]   = 45   # Yatay çatlak
steel_base[:, 180:183]   = 45   # Dikey çizgi
steel_base[60:120, 60:80] = 200 # Parlak nokta (yansıma)
# Tuz-biber gürültüsü
sp = np.random.rand(*steel_base.shape)
steel_base[sp < 0.03] = 0
steel_base[sp > 0.97] = 255

steel_img = cv2.cvtColor(steel_base, cv2.COLOR_GRAY2BGR)  # BGR forma çevir
steel_gray = cv2.cvtColor(steel_img, cv2.COLOR_BGR2GRAY)

# ADIM 1–3: Farklı blur yöntemleri uygula
result_gaussian   = cv2.GaussianBlur(steel_gray, (5, 5), 0)
result_median     = cv2.medianBlur(steel_gray, 5)
result_bilateral  = cv2.bilateralFilter(steel_gray, d=9, sigmaColor=75, sigmaSpace=75)
result_eq         = cv2.equalizeHist(steel_gray)
# Kombinasyon: blur + eşitleme
result_combo      = cv2.equalizeHist(cv2.GaussianBlur(steel_gray, (3, 3), 0))

# ADIM 4: Yan yana karşılaştır
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

panels = [
    (steel_gray,       'Orijinal (Gri)\n(gürültülü çelik yüzeyi)'),
    (result_gaussian,  'GaussianBlur (5,5)\nGenel gürültü'),
    (result_median,    'medianBlur (k=5)\nTuz-biber temizleme'),
    (result_bilateral, 'bilateralFilter\nd=9 sC=75 sS=75'),
    (result_eq,        'equalizeHist\nKontrast artırma'),
    (result_combo,     '★ Kombinasyon\nGaussian(3,3) + equalizeHist'),
]
for ax, (im, title) in zip(axes.flat, panels):
    ax.imshow(im, cmap='gray'); ax.set_title(title, fontsize=10); ax.axis('off')

plt.suptitle('Mini Proje 1: Çelik Yüzey Gürültü Azaltma', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ADIM 5: Seçimini yap ve kaydet
best = result_combo  # <-- DEĞİŞTİR: hangi sonuç daha iyi?
cv2.imwrite('/kaggle/working/mini1_best.jpg', best)

print("✅ Mini Proje 1 tamamlandı!")
print("\nDeğerlendirme notu:")
print("  GaussianBlur  → genel gürültü azaldı ama kenarlar biraz yumuşadı")
print("  medianBlur    → tuz-biber noktaları temizlendi, çizgiler korundu")
print("  bilateralFilter → kenarlar iyi korundu, gürültü azaldı")
print("  Kombinasyon   → kontrast arttı + hafif blur = en dengeli sonuç")

---
## ☕ Kısa Ara (10 dk) — Session 1 Sonu
---
## 📐 K-07 | ROI — Region of Interest
📌 *Slayt 27 — İlgi Bölgesi Seçimi*

In [ ]:
# ================================================================
# K-07: ROI (Region of Interest) — İlgi Bölgesi Kırpma
# ================================================================
#
# ROI = görüntünün sadece belirli bir dikdörtgen bölgesini kesme
#
# SÖZDIZIMI: img[y1:y2, x1:x2]
#   y1 = üst kenar (satır başlangıcı)
#   y2 = alt kenar (satır bitişi)
#   x1 = sol kenar (sütun başlangıcı)
#   x2 = sağ kenar (sütun bitişi)
#
# ⚠️ y (satır/yükseklik) ÖNCE gelir! Matematikteki (x,y) sırasının TERSİ!
#
# ⚠️ roi = img[...] REFERANSTIR, kopya değil!
#    roi değiştirilirse orijinal img de değişir!
#    Güvenli çalışma için: roi = img[...].copy()
#
# AVANTAJLAR:
#   ⚡ Hız: Küçük bölge = çok daha hızlı işlem
#   🎯 Doğruluk: Arka planı görmezden gel
#   📐 Stabilite: Algoritma daha tutarlı çalışır
#

# Konveyör bant simülasyonu
conveyor = np.ones((400, 640, 3), dtype=np.uint8) * 50  # Koyu arka plan
conveyor[170:250, :] = [80, 90, 80]    # Konveyör bandı
# Ürünler
for xp in [80, 250, 430]:
    conveyor[175:245, xp:xp+100] = [180, 160, 140]  # Ürün kutuları

# Statik ROI — konveyör bant bölgesi
x, y, w, h = 30, 165, 580, 90
roi_ref  = conveyor[y:y+h, x:x+w]         # Referans (dikkatli!)
roi_copy = conveyor[y:y+h, x:x+w].copy()  # Güvenli kopya

# ROI'yi görsel olarak işaretle
display = conveyor.copy()
cv2.rectangle(display, (x, y), (x+w, y+h), (0, 255, 0), 3)  # Yeşil çerçeve
cv2.putText(display, f'ROI {w}x{h}px', (x+5, y-8),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(display, cv2.COLOR_BGR2RGB))
axes[0].set_title('Konveyör — ROI Seçimi (yeşil kutu)')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(roi_copy, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'ROI Bölgesi — {roi_copy.shape}')
axes[1].axis('off')

plt.suptitle('K-07: ROI — Region of Interest', fontweight='bold')
plt.tight_layout()
plt.show()

full_px = conveyor.shape[0]*conveyor.shape[1]
roi_px  = roi_copy.shape[0]*roi_copy.shape[1]
print(f"Tam görüntü: {full_px:,} piksel")
print(f"ROI bölgesi: {roi_px:,} piksel")
print(f"İşlem hız kazancı: ~{full_px/roi_px:.1f}x daha hızlı")

---
## 🎨 K-08 | HSV Renk Filtreleme & cv2.inRange()
📌 *Slayt 28 — Renk Bazlı Nesne Seçimi*

In [ ]:
# ================================================================
# K-08: cv2.inRange() — HSV Renk Filtresi
# ================================================================
#
# cv2.inRange(src, lowerb, upperb)
#
#  src    → HSV görüntüsü (cvtColor ile BGR→HSV yapılmış olmalı!)
#           ⚠️ BGR görüntü verme — HSV eşikleri yanlış çalışır!
#
#  lowerb → Alt eşik: np.array([H_min, S_min, V_min])
#           H_min = minimum ton değeri (0-179)
#           S_min = minimum doygunluk (50 önerilen — soluk renkleri dışla)
#           V_min = minimum parlaklık (50 önerilen — karanlığı dışla)
#
#  upperb → Üst eşik: np.array([H_max, S_max, V_max])
#           genelde S_max=255, V_max=255
#
#  ÇIKTI → Binary mask: eşik içindeyse 255 (beyaz), dışındaysa 0 (siyah)
#
# cv2.bitwise_and(src1, src2, mask=mask)
#  → Mask'te 255 olan pikseller korunur, 0 olanlar siyah yapılır
#  → src1 ve src2 genelde aynı görüntü olur (orijinali filtrele)
#

# Test görüntüsü — farklı renkli bölgeler (K-01'den)
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)  # ← Önce HSV'ye çevir!

# --- Mavi renk filtresi ---
lower_blue = np.array([100, 60, 60])   # H_min=100, S_min=60, V_min=60
upper_blue = np.array([130, 255, 255]) # H_max=130
mask_blue  = cv2.inRange(hsv, lower_blue, upper_blue)  # Binary mask
result_blue = cv2.bitwise_and(img, img, mask=mask_blue)

# --- Yeşil renk filtresi ---
lower_green = np.array([35, 50, 50])    # <-- DEĞİŞTİR: farklı aralıklar dene
upper_green = np.array([77, 255, 255])
mask_green  = cv2.inRange(hsv, lower_green, upper_green)
result_green = cv2.bitwise_and(img, img, mask=mask_green)

# --- Sarı renk filtresi ---
lower_yellow = np.array([20, 50, 50])
upper_yellow = np.array([40, 255, 255])
mask_yellow  = cv2.inRange(hsv, lower_yellow, upper_yellow)
result_yellow = cv2.bitwise_and(img, img, mask=mask_yellow)

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
rows = [
    (img_rgb, mask_blue,   result_blue,   'Mavi Filtre',   'Mask (mavi)',  'Sonuç'),
    (img_rgb, mask_green,  result_green,  'Yeşil Filtre',  'Mask (yeşil)', 'Sonuç'),
    (img_rgb, mask_yellow, result_yellow, 'Sarı Filtre',   'Mask (sarı)',  'Sonuç'),
]
for i, (orig, mask, res, t1, t2, t3) in enumerate(rows):
    axes[i,0].imshow(orig);                         axes[i,0].set_title(t1); axes[i,0].axis('off')
    axes[i,1].imshow(mask, cmap='gray');            axes[i,1].set_title(t2); axes[i,1].axis('off')
    axes[i,2].imshow(cv2.cvtColor(res, cv2.COLOR_BGR2RGB)); axes[i,2].set_title(t3); axes[i,2].axis('off')

plt.suptitle('K-08: HSV inRange() — Renk Filtreleme', fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Mask değerleri: 255=filtre içinde (seçildi) | 0=filtre dışında (elendi)")

---
## 🔗 K-09 | Mask Birleştirme & Temel Morfoloji
📌 *Slayt 29 — bitwise_or, bitwise_and, MORPH_OPEN*

In [ ]:
# ================================================================
# K-09: Mask Birleştirme & Morfoloji
# ================================================================
#
# ── BİTWISE İŞLEMLER ─────────────────────────────────────────
#
# cv2.bitwise_and(src1, src2, mask=None)
#   → Her pikselde AND işlemi
#   → mask parametresiyle: mask=255 olan pikseller korunur
#   → Kullanım: Maskeli orijinal görüntü elde etmek için
#
# cv2.bitwise_or(src1, src2)
#   → Her pikselde OR işlemi
#   → İki maskeden UNION → ikisi de dahil
#   → Kullanım: Birden fazla renk aralığını birleştir
#
# cv2.bitwise_not(src)
#   → Maskeyi tersine çevirir: 255→0, 0→255
#   → Kullanım: Seçili bölgenin DIŞINI al
#
# ── MORFOLOJİK İŞLEMLER ─────────────────────────────────────
#
# cv2.morphologyEx(src, op, kernel)
#
#  op (işlem) :
#   cv2.MORPH_OPEN    = Erosion → Dilation
#                       Küçük gürültü noktaları SİLER (en çok kullanılan)
#
#   cv2.MORPH_CLOSE   = Dilation → Erosion
#                       Küçük boşlukları/delikleri DOLDURUR
#
#   cv2.MORPH_ERODE   = Sadece erosion  → beyaz bölgeyi küçültür
#   cv2.MORPH_DILATE  = Sadece dilation → beyaz bölgeyi büyütür
#
#  kernel : İşlem penceresi boyutu
#   np.ones((3,3), np.uint8)  = 3×3 kare kernel (hafif)
#   np.ones((5,5), np.uint8)  = 5×5 kare kernel (orta)   ← sık kullanılan
#   np.ones((9,9), np.uint8)  = 9×9 kare kernel (güçlü)
#

hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# İki farklı renk maskesi oluştur
mask_blue  = cv2.inRange(hsv, np.array([100, 60, 60]), np.array([130, 255, 255]))
mask_green = cv2.inRange(hsv, np.array([ 35, 50, 50]), np.array([ 77, 255, 255]))

# Birleştirme (OR): mavi VEYA yeşil
mask_union = cv2.bitwise_or(mask_blue, mask_green)

# Morfoloji: küçük gürültü noktalarını temizle (MORPH_OPEN)
kernel = np.ones((5, 5), np.uint8)  # <-- DEĞİŞTİR: (3,3) veya (9,9)
mask_clean = cv2.morphologyEx(mask_union, cv2.MORPH_OPEN, kernel)

# Boşluk doldurma (MORPH_CLOSE)
mask_filled = cv2.morphologyEx(mask_union, cv2.MORPH_CLOSE, kernel)

# Sonucu orijinale uygula
result = cv2.bitwise_and(img, img, mask=mask_clean)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
panels = [
    (mask_blue,   'Mask: Mavi',   'gray'),
    (mask_green,  'Mask: Yeşil',  'gray'),
    (mask_union,  'bitwise_OR\n(Mavi + Yeşil)', 'gray'),
    (mask_clean,  'MORPH_OPEN (5×5)\nGürültü temizlendi', 'gray'),
    (mask_filled, 'MORPH_CLOSE (5×5)\nBoşluklar dolduruldu', 'gray'),
    (cv2.cvtColor(result, cv2.COLOR_BGR2RGB), 'Son Sonuç\n(temiz mask ile)', None),
]
for ax, (im, title, cmap) in zip(axes.flat, panels):
    ax.imshow(im, cmap=cmap); ax.set_title(title, fontsize=9); ax.axis('off')

plt.suptitle('K-09: Mask Birleştirme & Morfoloji', fontweight='bold')
plt.tight_layout()
plt.show()

print("📌 Gün 2'de morfoloji tam kapsamlı işlenecek.")
print("   Bugün MORPH_OPEN ile gürültü temizlemeyi bilmek yeterli.")

---
## 🔨 Mini Proje 2 — Konveyörde ROI + Renk Filtresi
📌 *Slayt 30 — K-10 | Session 2 Uygulaması (40–50 dk)*

In [ ]:
# ================================================================
# MİNİ PROJE 2: Konveyör — ROI + HSV Renk Filtresi
# ================================================================

# Konveyör simülasyonu
conv = np.ones((350, 600, 3), dtype=np.uint8) * 60
conv[130:220, :] = [75, 85, 75]  # Konveyör bandı

# Ürünler — farklı renklerde
conv[140:210,  50:130]  = [0, 180,  0]   # Yeşil ürün
conv[140:210, 200:290]  = [0, 0,   200]  # Kırmızı ürün (NOK — yabancı madde)
conv[140:210, 350:430]  = [0, 180,  0]   # Yeşil ürün
conv[140:210, 490:560]  = [0, 180,  0]   # Yeşil ürün

# ADIM 1: Statik ROI belirle
rx, ry, rw, rh = 20, 125, 560, 100  # Konveyör bant bölgesi
roi = conv[ry:ry+rh, rx:rx+rw].copy()

# ADIM 2: BGR → HSV
roi_hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

# ADIM 3: Yeşil renk maskesi (normal ürün)
lower_ok = np.array([35, 60, 60])
upper_ok = np.array([77, 255, 255])
mask_ok = cv2.inRange(roi_hsv, lower_ok, upper_ok)

# ADIM 4: Kırmızı renk maskesi (NOK / yabancı madde)
lower_nok1 = np.array([0,  60,  60])
upper_nok1 = np.array([10, 255, 255])
lower_nok2 = np.array([160, 60,  60])
upper_nok2 = np.array([179, 255, 255])
mask_nok = cv2.bitwise_or(
    cv2.inRange(roi_hsv, lower_nok1, upper_nok1),
    cv2.inRange(roi_hsv, lower_nok2, upper_nok2)
)

# ADIM 5: Morfoloji — gürültü temizle
kernel = np.ones((5, 5), np.uint8)
mask_ok_clean  = cv2.morphologyEx(mask_ok,  cv2.MORPH_OPEN, kernel)
mask_nok_clean = cv2.morphologyEx(mask_nok, cv2.MORPH_OPEN, kernel)

# ADIM 6: Sonuç görseli (overlay)
result = cv2.bitwise_and(roi, roi, mask=mask_ok_clean)
nok_result = cv2.bitwise_and(roi, roi, mask=mask_nok_clean)

# ADIM 7: Görselleştir
full_display = conv.copy()
cv2.rectangle(full_display, (rx, ry), (rx+rw, ry+rh), (0, 255, 0), 2)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
panels = [
    (cv2.cvtColor(full_display, cv2.COLOR_BGR2RGB),   'Konveyör + ROI Kutusu'),
    (cv2.cvtColor(roi, cv2.COLOR_BGR2RGB),            'ROI Bölgesi'),
    (mask_ok_clean,                                    'Mask: Yeşil (OK Ürün)', ),
    (mask_nok_clean,                                   'Mask: Kırmızı (NOK!)'),
    (cv2.cvtColor(result, cv2.COLOR_BGR2RGB),         'Sonuç: OK Ürünler'),
    (cv2.cvtColor(nok_result, cv2.COLOR_BGR2RGB),     'Sonuç: NOK Ürünler'),
]
cmaps = [None, None, 'gray', 'gray', None, None]
for ax, (im, title), cmap in zip(axes.flat, panels, cmaps):
    ax.imshow(im, cmap=cmap); ax.set_title(title, fontsize=10); ax.axis('off')

plt.suptitle('Mini Proje 2: Konveyör ROI + Renk Filtresi (OK/NOK Ayrımı)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

ok_count  = cv2.countNonZero(mask_ok_clean)
nok_count = cv2.countNonZero(mask_nok_clean)
print(f"✅ OK ürün piksel alanı : {ok_count:,} px")
print(f"❌ NOK ürün piksel alanı: {nok_count:,} px")
print("\n✅ Mini Proje 2 tamamlandı!")

---
## ☕ Kısa Ara (10 dk) — Session 2 Sonu ★ YENİ
---
## 🔲 K-11 | Edge Detection — Canny / Sobel / Laplacian
📌 *Slayt 33–34 — Kenar Tespiti*

In [ ]:
# ================================================================
# K-11-A: cv2.Canny() — Parametre Detayları
# ================================================================
#
# cv2.Canny(image, threshold1, threshold2, apertureSize=3)
#
#  image      → uint8, 1 kanallı (gri) görüntü
#               ⚠️ BLUR UYGULANMIŞ OLMALI! Gürültü yanlış edge üretir!
#
#  threshold1 → Alt eşik (düşük)
#               Bu değerin ALTINDAKI gradientler → kenar DEĞİL
#               Düşür → daha fazla (ama gürültülü) edge
#               Artır → daha az (ama temiz) edge
#               Başlangıç önerisi: 50
#
#  threshold2 → Üst eşik (yüksek)
#               Bu değerin ÜSTÜNDEKİ gradientler → kesinlikle KENAR
#               KURAL: threshold2 ≈ 2×threshold1 veya 3×threshold1
#               Başlangıç önerisi: 150 (= 3×50)
#
#  İki eşik arası → Eğer kenar piksellerine bağlıysa dahil edilir
#
#  ÇIKTI: Binary görüntü — kenarlar beyaz (255), arkaplan siyah (0)
#
# NEDEN CANNY?
#   1. Önce blur ile gürültü azaltır (dahili Gaussian)
#   2. Gradient hesaplar (Sobel benzeri)
#   3. Non-maximum suppression → ince tek piksel genişliğinde kenar
#   4. Hysteresis thresholding → iki eşikle kesin/olası kenar ayrımı
#

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (5, 5), 0)  # Önce blur! (şart)

# Farklı eşik kombinasyonları
canny_50_150  = cv2.Canny(blurred, 50, 150)    # t2 = 3×t1  (başlangıç)  # <-- DEĞİŞTİR
canny_30_90   = cv2.Canny(blurred, 30,  90)    # Daha fazla edge
canny_100_200 = cv2.Canny(blurred, 100, 200)   # Daha az (temiz) edge

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (im, title) in zip(axes, [
    (gray,          'Orijinal Gri'),
    (canny_30_90,   'Canny t1=30 t2=90\nDaha fazla edge'),
    (canny_50_150,  'Canny t1=50 t2=150\nDengeli (öneri)'),
    (canny_100_200, 'Canny t1=100 t2=200\nDaha az edge'),
]):
    ax.imshow(im, cmap='gray'); ax.set_title(title); ax.axis('off')

plt.suptitle('K-11-A: cv2.Canny() Threshold Karşılaştırması', fontweight='bold')
plt.tight_layout()
plt.show()

print("🔑 Kural: t2 ≈ 2–3 × t1  |  Gürültülü görüntü → blur'u artır, sonra Canny dene")

In [ ]:
# ================================================================
# K-11-B: cv2.Sobel() & cv2.Laplacian() — Parametre Detayları
# ================================================================
#
# cv2.Sobel(src, ddepth, dx, dy, ksize=3)
#
#  src    → Giriş görüntüsü (gri ton)
#  ddepth → Çıktı veri tipi
#           cv2.CV_64F (öneri!) — negatif değerleri yakalar
#           cv2.CV_8U  — negatif değerleri 0 yapar, bilgi kaybı!
#  dx     → X yönü türevi: 1 = yatay kenarlar (| gibi dikey çizgiler)
#  dy     → Y yönü türevi: 1 = dikey kenarlar  (— gibi yatay çizgiler)
#           ⚠️ dx=1, dy=1 AYNI ANDA VERME! Önce (1,0) sonra (0,1)
#  ksize  → Sobel kernel: 1, 3, 5, 7  (varsayılan 3)
#           -1 → Scharr operatörü kullan (daha hassas)
#
# cv2.Laplacian(src, ddepth, ksize=1)
#
#  2. türev tabanlı — tüm yönlerde kenar bulur
#  Gürültüye hassas → blur önce şart!
#

# Sobel X (yatay kenarlar)
sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)  # dx=1, dy=0
sobel_x_abs = np.uint8(np.absolute(sobel_x))            # uint8'e çevir

# Sobel Y (dikey kenarlar)
sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)  # dx=0, dy=1
sobel_y_abs = np.uint8(np.absolute(sobel_y))

# Magnitude: X ve Y'yi birleştir
sobel_mag = cv2.magnitude(sobel_x, sobel_y)  # sqrt(X² + Y²)
sobel_mag = np.uint8(np.clip(sobel_mag, 0, 255))

# Laplacian
laplacian = cv2.Laplacian(gray, cv2.CV_64F, ksize=3)
laplacian_abs = np.uint8(np.absolute(laplacian))

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (im, title) in zip(axes, [
    (gray,          'Orijinal'),
    (sobel_x_abs,   'Sobel X\n(dx=1 dy=0)\nyatay kenarlar'),
    (sobel_y_abs,   'Sobel Y\n(dx=0 dy=1)\ndikey kenarlar'),
    (sobel_mag,     'Sobel Magnitude\n(X²+Y²)^0.5'),
    (laplacian_abs, 'Laplacian\n(2. türev, tüm yönler)'),
]):
    ax.imshow(im, cmap='gray'); ax.set_title(title, fontsize=9); ax.axis('off')

plt.suptitle('K-11-B: Sobel & Laplacian Karşılaştırması', fontweight='bold')
plt.tight_layout()
plt.show()

print("📊 Seçim Rehberi:")
print("  Genel kenar → Canny (gürültüye dayanıklı, ince kenar)")
print("  Kenar YÖNÜ  → Sobel X + Sobel Y ayrı")
print("  Hızlı analiz → Laplacian (ama gürültüye hassas)")

---
## 🔷 K-12 | Contour Analizi — findContours()
📌 *Slayt 35 — Şekil Çıkarımı ve Özellik Hesabı*

In [ ]:
# ================================================================
# K-12: cv2.findContours() — Contour Analizi Rehberi
# ================================================================
#
# cv2.findContours(image, mode, method)
#
#  image  → Binary görüntü (Canny çıktısı veya threshold sonrası)
#           ⚠️ Renkli görüntü verme! Binary (0/255) olmalı!
#
#  mode   → Hangi konturlar bulunacak:
#           cv2.RETR_EXTERNAL  → Sadece en dış konturlar
#                                Nesne sınırları için yeterli (öneri)
#           cv2.RETR_LIST      → Tüm konturlar (hiyerarşi yok)
#           cv2.RETR_TREE      → Tüm konturlar + hiyerarşi bilgisi
#
#  method → Kontur noktalarını nasıl sakla:
#           cv2.CHAIN_APPROX_SIMPLE  → Sadece köşe noktaları
#                                      Bellek verimli (öneri)
#           cv2.CHAIN_APPROX_NONE    → Tüm noktalar
#                                      Hassas ama büyük bellek
#
# ÇIKTI: (contours, hierarchy)
#   contours → Her kontur: N×1×2 boyutlu NumPy array (N nokta)
#   hierarchy → Konturlar arası ilişki (RETR_TREE için önemli)
#
# KONTUR ÖZELLİKLERİ:
#
#   cv2.contourArea(cnt)      → Piksel cinsinden alan
#                               Küçük gürültü → if area < 500: continue
#
#   cv2.arcLength(cnt, True)  → Çevre uzunluğu (piksel)
#                               True = kapalı eğri
#                               False = açık eğri
#
#   cv2.moments(cnt)          → İstatistiksel momentler
#                               cx = M['m10']/M['m00']  (x merkezi)
#                               cy = M['m01']/M['m00']  (y merkezi)
#
#   cv2.boundingRect(cnt)     → Dik sınır kutusu → (x, y, w, h)
#
#   cv2.minAreaRect(cnt)      → Döndürülmüş minimum kutu (dönel nesneler)
#
#   cv2.minEnclosingCircle(cnt) → Minimum kapsayan çember → (x,y), r
#

# Test görüntüsü: birkaç geometrik şekil
shape_img = np.zeros((300, 500, 3), dtype=np.uint8)
cv2.rectangle(shape_img, (30, 40), (130, 140), (255, 255, 255), -1)   # Kare
cv2.circle(shape_img, (250, 90), 60, (200, 200, 200), -1)              # Daire
cv2.ellipse(shape_img, (400, 90), (70, 40), 0, 0, 360, (180, 180, 180), -1)  # Elips
cv2.rectangle(shape_img, (50, 180), (200, 260), (160, 160, 160), -1)  # Dikdörtgen

# Canny edge detection
gray_s = cv2.cvtColor(shape_img, cv2.COLOR_BGR2GRAY)
blur_s = cv2.GaussianBlur(gray_s, (5, 5), 0)
edges_s = cv2.Canny(blur_s, 30, 100)

# findContours
contours, hierarchy = cv2.findContours(
    edges_s,
    cv2.RETR_EXTERNAL,       # Sadece dış konturlar
    cv2.CHAIN_APPROX_SIMPLE  # Sadece köşe noktaları
)

print(f"Bulunan kontur sayısı: {len(contours)}")

# Her konturu analiz et
result_img = shape_img.copy()
print("\nKontur özellikleri:")
print(f"{'No':<4} {'Alan':>8} {'Çevre':>8} {'cx':>6} {'cy':>6} {'w':>6} {'h':>6}")
print("-" * 48)

colors_draw = [(0,255,0), (0,165,255), (255,0,0), (0,0,255), (255,255,0)]

for i, cnt in enumerate(contours):
    area  = cv2.contourArea(cnt)            # Alan (piksel²)
    if area < 100: continue                  # Küçük gürültü atla
    peri  = cv2.arcLength(cnt, True)        # Çevre (piksel)
    M     = cv2.moments(cnt)               # Momentler
    cx    = int(M['m10']/M['m00']) if M['m00'] > 0 else 0  # x merkez
    cy    = int(M['m01']/M['m00']) if M['m00'] > 0 else 0  # y merkez
    x,y,w,h = cv2.boundingRect(cnt)        # Sınır kutusu

    color = colors_draw[i % len(colors_draw)]
    cv2.drawContours(result_img, [cnt], -1, color, 2)      # Konturu çiz
    cv2.rectangle(result_img, (x,y), (x+w,y+h), color, 1) # Kutuyu çiz
    cv2.circle(result_img, (cx,cy), 5, color, -1)          # Merkezi işaretle
    cv2.putText(result_img, f'#{i}', (x, y-5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    print(f"#{i:<3} {area:>8.0f} {peri:>8.1f} {cx:>6} {cy:>6} {w:>6} {h:>6}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(shape_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Orijinal'); axes[0].axis('off')
axes[1].imshow(edges_s, cmap='gray')
axes[1].set_title('Canny Edges'); axes[1].axis('off')
axes[2].imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
axes[2].set_title('Konturlar + BBox + Merkez'); axes[2].axis('off')

plt.suptitle('K-12: findContours() Analizi', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔨 Mini Proje 3 — Çelik Plaka Kenar Tespiti
📌 *Slayt 36 — K-13 | Session 3 Uygulaması (25–30 dk)*

In [ ]:
# ================================================================
# MİNİ PROJE 3: Çelik Plaka Kenar + Boyut Ölçümü (Piksel)
# ================================================================

# Çelik plaka simülasyonu
plate = np.ones((400, 600, 3), dtype=np.uint8) * 40  # Koyu arka plan

# Ana plaka (hafif gürültülü metal yüzeyi)
plate_region = np.random.normal(160, 20, (220, 380, 3)).clip(0, 255).astype(np.uint8)
plate[100:320, 110:490] = plate_region

# Yüzey kusurları (defect benzeri)
plate[150:156, 150:400] = 50   # Yatay çizgi
plate[250:253, 200:350] = 230  # Parlak bant
plate[180:220, 300:315] = 40   # Dikey çizgi

print("Plaka görüntüsü oluşturuldu:", plate.shape)

# PIPELINE:
# Adım 1: Gri dönüşüm
plate_gray = cv2.cvtColor(plate, cv2.COLOR_BGR2GRAY)

# Adım 2: Blur
plate_blur = cv2.GaussianBlur(plate_gray, (7, 7), 0)  # <-- DEĞİŞTİR: (3,3), (11,11)

# Adım 3: Canny
plate_edges = cv2.Canny(plate_blur, 30, 90)  # <-- DEĞİŞTİR: threshold'u ayarla

# Adım 4: findContours
contours, _ = cv2.findContours(plate_edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
print(f"Bulunan kontur sayısı: {len(contours)}")

# Adım 5: Büyük konturları filtrele (küçük gürültüleri at)
MIN_AREA = 5000  # <-- DEĞİŞTİR: görüntüne göre ayarla
filtered = [c for c in contours if cv2.contourArea(c) > MIN_AREA]
print(f"Filtre sonrası (alan > {MIN_AREA}): {len(filtered)} kontur")

# Adım 6: En büyük konturu seç (ana plaka sınırı)
result_plate = plate.copy()
measurements = []

for i, cnt in enumerate(sorted(filtered, key=cv2.contourArea, reverse=True)[:3]):
    area         = cv2.contourArea(cnt)
    x, y, w, h   = cv2.boundingRect(cnt)
    M            = cv2.moments(cnt)
    cx = int(M['m10']/M['m00']) if M['m00'] > 0 else x+w//2
    cy = int(M['m01']/M['m00']) if M['m00'] > 0 else y+h//2

    color = [(0,255,0), (0,165,255), (255,0,0)][i]
    cv2.rectangle(result_plate, (x,y), (x+w,y+h), color, 2)
    cv2.drawContours(result_plate, [cnt], -1, color, 2)
    cv2.circle(result_plate, (cx,cy), 7, color, -1)
    # Boyut yazısı
    cv2.putText(result_plate, f'{w}x{h}px',
                (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    measurements.append((i, area, w, h, cx, cy))

# Görselleştirme
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(plate, cv2.COLOR_BGR2RGB))
axes[0].set_title('Orijinal Çelik Plaka'); axes[0].axis('off')
axes[1].imshow(plate_edges, cmap='gray')
axes[1].set_title(f'Canny Edges (t1=30, t2=90)'); axes[1].axis('off')
axes[2].imshow(cv2.cvtColor(result_plate, cv2.COLOR_BGR2RGB))
axes[2].set_title('Sonuç: Sınır Kutusu + Boyutlar'); axes[2].axis('off')

plt.suptitle('Mini Proje 3: Çelik Plaka Kenar Tespiti & Piksel Ölçümü', fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📏 Ölçüm Sonuçları:")
for i, area, w, h, cx, cy in measurements:
    print(f"  #{i}: Alan={area:.0f}px²  Genişlik={w}px  Yükseklik={h}px  Merkez=({cx},{cy})")

cv2.imwrite('/kaggle/working/mini3_result.jpg', result_plate)
print("\n✅ Mini Proje 3 tamamlandı!")
print("📌 Gerçek ölçüm (mm) için Gün 5'te kamera kalibrasyonu yapacağız.")

---
## 🐛 K-14 | Debugging Checklist
📌 *Slayt 37 — Pipeline Hata Ayıklama*

In [ ]:
# ================================================================
# K-14: Debug Yardımcı Fonksiyonları
# ================================================================
#
# Altın Kural: Her adımı görselleştir!
#   giriş → gray → blur → mask/edges → final overlay
#

def show_pipeline(steps_dict, cols=4):
    """
    Pipeline adımlarını yan yana göster.
    
    Kullanım:
        show_pipeline({
            'Orijinal': img,
            'Gray': gray,
            'Blur': blurred,
            'Edges': edges,
        })
    """
    n = len(steps_dict)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.5))
    if rows == 1: axes = axes.reshape(1, -1)

    for idx, (title, im) in enumerate(steps_dict.items()):
        ax = axes[idx // cols][idx % cols]
        if im is None:
            ax.text(0.5, 0.5, 'None!', color='red', ha='center', transform=ax.transAxes)
        elif len(im.shape) == 3:
            ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(im, cmap='gray')
        ax.set_title(f'{title}\n{im.shape if im is not None else "-"}', fontsize=9)
        ax.axis('off')

    # Boş aksları gizle
    for idx in range(n, rows * cols):
        axes[idx // cols][idx % cols].axis('off')

    plt.tight_layout()
    plt.show()


def debug_image(im, name='img'):
    """Görüntü hakkında temel bilgileri yazdır."""
    print(f"[{name}]")
    if im is None:
        print("  ⚠️  None! → Dosya okunamadı veya işlem başarısız")
        return
    print(f"  shape : {im.shape}")
    print(f"  dtype : {im.dtype}")
    print(f"  min   : {im.min()}  |  max : {im.max()}  |  mean : {im.mean():.1f}")
    if im.dtype == np.float32 or im.dtype == np.float64:
        print("  ⚠️  float! → imshow için normalize et veya uint8'e çevir")


# Tam pipeline örneği — debug fonksiyonlarıyla
sample = img.copy()

step_gray   = cv2.cvtColor(sample, cv2.COLOR_BGR2GRAY)
step_blur   = cv2.GaussianBlur(step_gray, (5, 5), 0)
step_edges  = cv2.Canny(step_blur, 50, 150)
contours_d, _ = cv2.findContours(step_edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
step_result = sample.copy()
cv2.drawContours(step_result, contours_d, -1, (0, 255, 0), 2)

# Debug bilgisi
for name, im in [('Orijinal', sample), ('Gray', step_gray),
                  ('Blur', step_blur), ('Edges', step_edges)]:
    debug_image(im, name)
    print()

# Pipeline görselleştirme
show_pipeline({
    'Giriş (BGR)':      sample,
    'Gri Ton':          step_gray,
    'GaussianBlur(5,5)': step_blur,
    'Canny (50,150)':   step_edges,
    'Sonuç (Contours)': step_result,
})

In [ ]:
# ================================================================
# K-14: Sık Yapılan Hatalar & Düzeltmeleri
# ================================================================

print("HATA → NEDEN → DÜZELTME")
print("=" * 60)

hatas = [
    ("❌ Renkler ters (sarı↔mavi)",
     "BGR/RGB karışması",
     "cv2.cvtColor(img, cv2.COLOR_BGR2RGB) ekle"),

    ("❌ Siyah ekran veya beyaz",
     "dtype karışması (float vs uint8)",
     "(arr*255).astype(np.uint8) veya np.clip ile düzelt"),

    ("❌ Mask hiçbir şey tutmuyor",
     "HSV eşik aralıkları yanlış",
     "trackbar ile canlı dene veya debug_image ile H değerini kontrol et"),

    ("❌ imread None döndü",
     "Dosya yolu yanlış veya bozuk",
     "os.path.exists(path) ile kontrol et, assert ekle"),

    ("❌ Canny çok gürültülü",
     "Blur uygulanmamış",
     "GaussianBlur(gray, (5,5), 0) → Canny sırasına dikkat"),

    ("❌ findContours hata",
     "Binary görüntü değil",
     "Canny veya threshold çıktısına findContours uygula"),

    ("❌ ROI değişince orijinal bozuluyor",
     "Referans kopyası (ROI = view)",
     "roi = img[...].copy() ile bağımsız kopya al"),

    ("❌ Kernel boyutu hatası",
     "GaussianBlur'a çift sayı verildi",
     "ksize her zaman tek sayı: (3,3) ✓  (4,4) ✗"),
]

for hata, neden, duzeltme in hatas:
    print(f"\n{hata}")
    print(f"  Neden   : {neden}")
    print(f"  Çözüm   : {duzeltme}")

print("\n" + "=" * 60)
print("✅ Debug Altın Kuralı: Her adımı görselleştir!")
print("   show_pipeline() fonksiyonunu kullan")

---
## 📋 Gün 1 Özeti & Kontrol Listesi

### ✅ Öğrenilen Fonksiyonlar

| Fonksiyon | Ne işe yarar? | Kritik parametre |
|---|---|---|
| `cv2.imread(path, flags)` | Görüntü okuma | flags=0 → gri, 1 → BGR |
| `cv2.imwrite(path, img, params)` | Görüntü kaydetme | JPEG_QUALITY |
| `cv2.cvtColor(img, code)` | Renk dönüşümü | COLOR_BGR2HSV |
| `cv2.GaussianBlur(img, ksize, sigmaX)` | Gürültü azaltma | ksize = tek sayı! |
| `cv2.medianBlur(img, ksize)` | Tuz-biber gürültüsü | ksize = tek sayı! |
| `cv2.bilateralFilter(img, d, sC, sS)` | Kenar koruyucu blur | d=9, sC=75 |
| `cv2.equalizeHist(gray)` | Kontrast artırma | sadece gri! |
| `cv2.inRange(hsv, lower, upper)` | Renk maskesi | HSV H=0-179 |
| `cv2.bitwise_and/or(src1, src2, mask)` | Mask işlemleri | mask parametresi |
| `cv2.morphologyEx(img, op, kernel)` | Morfoloji | MORPH_OPEN |
| `cv2.Canny(blur, t1, t2)` | Edge detection | t2 ≈ 3×t1 |
| `cv2.Sobel(gray, CV_64F, dx, dy)` | Gradient kenar | dx veya dy, ikisi aynı anda değil |
| `cv2.findContours(edges, mode, method)` | Kontur bulma | RETR_EXTERNAL |
| `cv2.contourArea(cnt)` | Alan hesabı | gürültü filtresi için |
| `cv2.boundingRect(cnt)` | Sınır kutusu | → x,y,w,h |

### 🎯 Gün 2
- 📌 CLAHE (adaptif histogram eşitleme) — equalizeHist'ten daha iyi!
- 📌 Arka plan silme (MOG2, KNN, GrabCut)
- 📌 Perspektif düzeltme ve geometrik dönüşümler
- 📌 Morfoloji (Gün 1'den daha derin)
- 📌 Roboflow ile annotation
- 📌 Data augmentation


In [ ]:
# ================================================================
# Tüm çıktıları kaydet
# ================================================================

import os

saved = [
    '/kaggle/working/mini1_best.jpg',
    '/kaggle/working/mini3_result.jpg',
    '/kaggle/working/kalite_95.jpg',
    '/kaggle/working/img.png',
]

print("Kaydedilen çıktılar:")
for f in saved:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024
        print(f"  ✅ {os.path.basename(f):<30} {size:.1f} KB")
    else:
        print(f"  ⚠️  {os.path.basename(f):<30} (henüz oluşturulmadı)")

print("\n🏆 Gün 1 Notebook tamamlandı!")
print("   Tüm K-0X bölümleri işlendi.")
print("   Notebook'u kaydet: File → Save Version")